# Análise de sensibilidade dos parâmetros do pipeline

Este notebook responde à solicitação de análise de sensibilidade do artigo. O
objetivo não é otimizar o pipeline nem escolher parâmetros após observar o
conjunto de teste. A análise altera **um parâmetro por vez** no conjunto de
validação e mede duas saídas principais:

- taxa de sucesso na localização dos óstios, considerando `both_correct` e
  `both_tolerable` como sucesso;
- Dice Score da segmentação coronariana.

A configuração de referência reproduz o run `train/normal_rg/2026-06-29_09-42-27`:
percentil superior 99.7, restrição em z de 40 mm, divisor global 7, fração
mínima de vesselness 7.8% e fator de relaxamento 0.98. Os níveis avaliados são:

- percentil superior: 99.5, 99.7 e 99.9;
- restrição em z: 30, 40 e 50 mm;
- divisor do Region Growing: 5, 7 e 9;
- fração mínima de vesselness: 5%, 7.8% e 9%, mantendo o fator 0.98.

O conjunto de teste não participa desta análise. As figuras 3D mostram um bom
resultado e erros representativos da configuração de referência.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT_CANDIDATES = (CURRENT_DIR, *CURRENT_DIR.parents)
REPO_ROOT = next(
    root for root in REPO_ROOT_CANDIDATES if (root / "src" / "utils").is_dir()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    run_qualitative_pipeline_case,
    select_parameter_validation_cases,
)
from utils.experiments.fuzzy_pipeline_comparison import split_overrides  # noqa: E402
from utils.experiments.sweep_common import apply_overrides  # noqa: E402
from utils.project.config import scale_config_to_resolution  # noqa: E402
from utils.project.notebook_env import resolve_imagecas_base_path  # noqa: E402
from utils.visualization import visualize_aorta_ostia_artery  # noqa: E402

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## Configuração

Por padrão, o notebook abre a execução mais recente. Defina `RUN_NAME` para
fixar uma pasta específica. O alvo 0.58 representa a faixa solicitada de
56%-60% de Dice.

In [ ]:
ANALYSIS_DIR = REPO_ROOT / "output/segmentation/analysis/pipeline_parameter_validation"
RUNS_DIR = ANALYSIS_DIR / "runs"
RUN_NAME = "sensitivity_normal_rg_val_30"
TARGET_DICE = 0.58

try:
    IMAGECAS_PATH = resolve_imagecas_base_path()
except FileNotFoundError:
    IMAGECAS_PATH = None

## Execução do experimento

Para executar as nove configurações em 30 imagens fixas de validação:

```bash
uv run python src/experiments/pipeline_parameter_validation.py \
  --split val \
  --sample-size 30 \
  --resolution mid \
  --gpu \
  --run-name sensitivity_normal_rg_val_30
```

O experimento executa a referência e oito perturbações OFAT. Os resultados são
salvos após cada variante. Caso o artigo reporte uma amostra de 30 exames, essa
escolha e os IDs devem ser declarados na metodologia; para uma evidência mais
forte, a mesma análise pode ser repetida em todo o conjunto de validação.

## Carregamento dos resultados

In [ ]:
def find_run_dir(run_name=None):
    if run_name:
        candidate = RUNS_DIR / run_name
        if not candidate.is_dir():
            raise FileNotFoundError(f"Run não encontrado: {candidate}")
        return candidate

    candidates = [
        path
        for path in RUNS_DIR.glob("*")
        if (path / "summary/ranking.csv").is_file()
        and (path / "results/image_results.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(
            "Nenhum run completo encontrado. Execute o comando da célula anterior."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


RUN_DIR = find_run_dir(RUN_NAME)
ranking_df = pd.read_csv(RUN_DIR / "summary/ranking.csv")
results_df = pd.read_csv(RUN_DIR / "results/image_results.csv")
parameters_df = pd.read_csv(RUN_DIR / "parameters/variant_parameters.csv")
run_config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))

if run_config.get("split") != "val" or set(results_df["split"].dropna()) != {"val"}:
    raise ValueError("Este notebook aceita apenas resultados do split de validação.")

print(f"Run carregado: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Imagens: {results_df['IMG_ID'].nunique()} | Variantes: {results_df['variant'].nunique()}")

## Configuração de referência

A referência foi congelada em `config/article_sensitivity_reference.json` a
partir do run `train/normal_rg/2026-06-29_09-42-27`. Ela preserva o threshold inferior
fixo, a interpolação dos círculos, `lower_fraction=0.85` e o RG com refinamento
de sementes e multiseed local. Somente os quatro parâmetros declarados na
análise OFAT são alterados.

In [ ]:
base_config = run_config["effective_base_config"]
reference_variant = next(item for item in run_config["variants"] if item["name"] == "baseline")
reference = reference_variant["overrides"]

baseline_values = pd.DataFrame(
    [
        ("Upper percentile", reference["MAX_THRESHOLD_PERCENTILE"], "99.5, 99.7, 99.9"),
        (
            "Limite z do segundo óstio (mm)",
            reference["OSTIA_DETECTION.max_z_diff_mm"],
            "30, 40, 50",
        ),
        (
            "Divisor global do RG",
            reference["REGION_GROWING.threshold_divisor"],
            "5, 7, 9",
        ),
        (
            "Fração mínima de vesselness",
            reference["REGION_GROWING.min_vesselness_fraction"],
            "0.05, 0.078, 0.09",
        ),
        (
            "Fator de relaxamento do piso",
            reference["REGION_GROWING.relaxed_floor_factor"],
            "0.98 (fixo)",
        ),
        ("Threshold inferior (HU)", base_config["MIN_THRESHOLD"], "-300 (fixo)"),
        (
            "Sigmas do vesselness da aorta",
            base_config["VESSELNESS_AORTA"]["sigmas"],
            "[2.5, 3.0] (fixo)",
        ),
    ],
    columns=["parâmetro", "referência canônica", "níveis avaliados"],
)
baseline_values

## Resultados quantitativos

A tabela abaixo contém diretamente as medidas relevantes para a análise de
sensibilidade: número e taxa de sucessos dos óstios, além de média, desvio
padrão, mediana, mínimo e máximo do Dice. O sucesso aceita localizações
classificadas como corretas ou toleráveis.

In [ ]:
def as_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype("string").str.strip().str.lower().isin(
        {"1", "true", "yes", "sim"}
    )


analysis_df = results_df.copy()
analysis_df["dice_observed"] = pd.to_numeric(
    analysis_df["dice_artery"], errors="coerce"
)
# Falhas completas recebem Dice zero, como nos resultados canônicos.
analysis_df["dice_artery"] = analysis_df["dice_observed"].fillna(0.0)
analysis_df["pipeline_failure"] = analysis_df["error"].notna()
analysis_df["ostia_success"] = as_boolean(analysis_df["ostia_success"])

sensitivity_summary = (
    analysis_df.groupby("variant", as_index=False)
    .agg(
        images=("IMG_ID", "nunique"),
        ostia_success_count=("ostia_success", "sum"),
        ostia_success_rate=("ostia_success", "mean"),
        dice_outputs=("dice_observed", "count"),
        pipeline_failures=("pipeline_failure", "sum"),
        mean_dice=("dice_artery", "mean"),
        std_dice=("dice_artery", "std"),
        median_dice=("dice_artery", "median"),
        min_dice=("dice_artery", "min"),
        max_dice=("dice_artery", "max"),
    )
    .merge(parameters_df[["variant", "parameter_group", "description"]], on="variant", how="left")
)
sensitivity_summary["ostia_success_percent"] = 100 * sensitivity_summary["ostia_success_rate"]
sensitivity_summary = sensitivity_summary.sort_values(
    ["parameter_group", "variant"],
    kind="stable",
)

summary_columns = [
    "variant",
    "parameter_group",
    "description",
    "images",
    "ostia_success_count",
    "ostia_success_percent",
    "dice_outputs",
    "pipeline_failures",
    "mean_dice",
    "std_dice",
    "median_dice",
    "min_dice",
    "max_dice",
]
sensitivity_summary[summary_columns]

### Dice médio e variabilidade por configuração

In [ ]:
plot_df = sensitivity_summary.sort_values("mean_dice", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["mean_dice"],
    xerr=plot_df["std_dice"].fillna(0),
    color="#2878B5",
    alpha=0.9,
    capsize=3,
)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
ax.set_xlabel("Dice Score médio", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

### Taxa de sucesso na localização dos óstios

In [ ]:
plot_df = sensitivity_summary.sort_values("ostia_success_percent", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["ostia_success_percent"],
    color="#3A923A",
    alpha=0.9,
)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=10)
ax.set_xlabel("Óstios corretos ou toleráveis (%)", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 100)
fig.tight_layout()
plt.show()

## Alterações pareadas em relação à referência

Como todas as configurações usam os mesmos IDs, a diferença de Dice é calculada
exame a exame. Essa tabela ajuda a distinguir uma mudança sistemática de uma
média dominada por poucos casos.

In [ ]:
baseline = results_df.loc[results_df["variant"] == "baseline", ["IMG_ID", "dice_artery"]].rename(
    columns={"dice_artery": "dice_baseline"}
)
pairwise_rows = []
for variant_name, variant_df in results_df.groupby("variant"):
    if variant_name == "baseline":
        continue
    paired = baseline.merge(
        variant_df[["IMG_ID", "dice_artery"]].rename(columns={"dice_artery": "dice_variant"}),
        on="IMG_ID",
        how="inner",
    ).dropna()
    delta = paired["dice_variant"] - paired["dice_baseline"]
    pairwise_rows.append(
        {
            "variant": variant_name,
            "paired_images": len(paired),
            "mean_delta_dice": delta.mean(),
            "median_delta_dice": delta.median(),
            "improved_images": int((delta > 0).sum()),
            "unchanged_images": int(np.isclose(delta, 0).sum()),
            "worse_images": int((delta < 0).sum()),
        }
    )

pairwise_df = pd.DataFrame(pairwise_rows).sort_values("mean_delta_dice", ascending=False)
pairwise_df

## Seleção dos casos qualitativos

Os casos são escolhidos na configuração de referência do artigo. Isso evita
usar uma variante favorecida pelos próprios resultados da validação para
ilustrar o desempenho qualitativo.

In [ ]:
QUALITATIVE_VARIANT = "baseline"

selected_cases = select_parameter_validation_cases(
    results_df,
    QUALITATIVE_VARIANT,
    target_dice=TARGET_DICE,
)
case_columns = [
    "case_type",
    "IMG_ID",
    "dice_artery",
    "ostia_status",
    "left_dist_mm",
    "right_dist_mm",
    "aorta_volume_fraction",
    "artery_volume_ratio",
    "selection_reason",
]
print(f"Configuração usada nas figuras: {QUALITATIVE_VARIANT}")
selected_cases.reindex(columns=case_columns)

## Preparação das visualizações 3D

Cada célula abaixo reexecuta somente um exame da referência. A aorta é mostrada
em vermelho, a artéria de referência em verde, a predição em azul e os óstios
em amarelo/ciano. As visualizações interativas são exibidas somente nas
células do notebook.

In [ ]:
variant_definitions = {item["name"]: item for item in run_config["variants"]}
qualitative_definition = variant_definitions[QUALITATIVE_VARIANT]
config_overrides, _ = split_overrides(qualitative_definition["overrides"])
qualitative_config = scale_config_to_resolution(
    apply_overrides(run_config["effective_base_config"], config_overrides)
)


def render_case(case_type):
    if IMAGECAS_PATH is None:
        raise FileNotFoundError(
            "Defina IMAGECAS_BASE_PATH para executar as visualizações qualitativas."
        )
    matches = selected_cases.loc[selected_cases["case_type"] == case_type]
    if matches.empty:
        print(f"Nenhum caso disponível para: {case_type}")
        return None

    selected = matches.iloc[0]
    image_id = int(selected["IMG_ID"])
    try:
        result = run_qualitative_pipeline_case(
            image_id,
            qualitative_config,
            IMAGECAS_PATH,
        )
    except Exception as exc:
        print(f"Não foi possível reconstruir IMG_ID={image_id}: {type(exc).__name__}: {exc}")
        return None

    _ = visualize_aorta_ostia_artery(
        result["aorta_mask"],
        result["ostia_left"],
        result["ostia_right"],
        artery_mask=result["artery_mask"],
        label_artery=result["label_artery"],
        spacing=result["scaled_spacing"],
        save_html_path=None,
        display_plot=True,
        plot_name=f"{case_type} | {QUALITATIVE_VARIANT} | IMG_ID={image_id}",
    )
    print(
        f"IMG_ID={image_id} | Dice={selected['dice_artery']:.4f} | "
        f"óstios={selected['ostia_status']}"
    )
    return result

### Caso com Dice alto

In [ ]:
high_dice_result = render_case("high_dice")

### Caso próximo de 56%-60% de Dice

In [ ]:
near_mean_result = render_case("near_target_mean")

### Falha na localização dos óstios

In [ ]:
ostia_failure_result = render_case("ostia_failure")

### Possível vazamento da máscara da aorta

In [ ]:
aorta_leak_result = render_case("suspected_aorta_leak")

### Falha ou vazamento na segmentação arterial

In [ ]:
segmentation_failure_result = render_case("segmentation_failure")

## Interpretação para o artigo

- A sensibilidade é pequena quando perturbar um parâmetro produz pouca mudança
  no Dice médio e na taxa de sucesso dos óstios.
- Resultados pareados mostram quantos exames melhoraram ou pioraram, evitando
  interpretar apenas a média agregada.
- As imagens 3D devem acompanhar a tabela quantitativa com um caso de bom
  desempenho e exemplos dos erros mais comuns: falha dos óstios, vazamento da
  aorta e falha da segmentação arterial.
- Esta análise pertence à validação. O conjunto de teste deve permanecer
  reservado para a avaliação final da configuração já definida.